In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM,GRU, Bidirectional, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences


In [6]:
texts = ['I love this product', 'This product is very trrible', 'Amazing service', 'Worst experience ever', 'I will buy this again', 'Not worth the money']
labels = np.array([1, 0, 1, 0, 1, 0])
tok = Tokenizer(num_words=100)
tok.fit_on_texts(texts)
seqs = pad_sequences(tok.texts_to_sequences(texts), maxlen=5)
seqs

array([[ 0,  2,  4,  1,  3],
       [ 1,  3,  5,  6,  7],
       [ 0,  0,  0,  8,  9],
       [ 0,  0, 10, 11, 12],
       [ 2, 13, 14,  1, 15],
       [ 0, 16, 17, 18, 19]], dtype=int32)

In [8]:
model_bi = Sequential([
    Embedding(input_dim=100, output_dim=8, input_length=5),
    Bidirectional(LSTM(16)),
    Dense(1, activation='sigmoid')
])
model_bi.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_bi.fit(seqs, labels.reshape(-1,1), epochs=100, verbose=0)  #verbose=0 to suppress output
test_texts = ['this product is amazing']
text_seq = pad_sequences(tok.texts_to_sequences(test_texts), maxlen=5)
prediction = model_bi.predict(text_seq)[0][0]
print(f'BI_directionl LSTM Output:')
print(f'Input Text: "{test_texts[0]}"')
print(f'Raw Score: {prediction:.4f}')
if prediction > 0.5:
    sentiment = 'Positive'
    feelings = 'Good'
else:
    sentiment = 'Negative'
    feelings = 'Worst'
print(f'The review is: {sentiment} ;sentiment and the customer is feeling ({feelings})')


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 878ms/step
BI_directionl LSTM Output:
Input Text: "this product is amazing"
Raw Score: 0.6754
The review is: Positive ;sentiment and the customer is feeling (Good)


In [16]:
test_texts = ['the battery life is greater']
text_seq = pad_sequences(tok.texts_to_sequences(test_texts), maxlen=5)
model_lstm = Sequential ([
    Embedding(input_dim= 100, output_dim=8, input_length=5),
    LSTM(4)
])
lstm_output = model_lstm.predict(text_seq)
print("\n------LSTM Hidden State (Memory Vector)-------")
print(f"input sentence: {test_texts[0]}")
print("--"* 40)
features = ["Product Quality", "Urgency", "Technical Issues", "User Satisfaction"]
for i in range(len(features)):
    strength = abs(lstm_output[0][i]) 
    print(f"{features[i]}: Strength = {strength:.4f}")
print("\n Note: Ina trained Model, 'User Satisfaction' would have much high value")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 557ms/step

------LSTM Hidden State (Memory Vector)-------
input sentence: the battery life is greater
--------------------------------------------------------------------------------
Product Quality: Strength = 0.0014
Urgency: Strength = 0.0052
Technical Issues: Strength = 0.0007
User Satisfaction: Strength = 0.0001

 Note: Ina trained Model, 'User Satisfaction' would have much high value


In [6]:
X_temp = np.array([
    [[20], [21], [22]],
    [[25], [26], [27]],
    [[30], [31], [32]]
])
y_temp = np.array([23,28,33])
model_gru = Sequential([
    GRU(32, input_shape=(3,1)),
    Dense(1)
])
model_gru.compile(optimizer='adam', loss='mse')
model_gru.fit(X_temp, y_temp, epochs=200, verbose=0)
test_input = np.array([[35, 36, 37]]).reshape(1, 3, 1)
prediction_temp = model_gru.predict(test_input)[0][0]

print("\n------GRU Prediction-------")
print(f"Input Sequence (Past 3 Days): {[35, 36, 37]}")
print(f"Model's Predicted Temperature: {prediction_temp:.2f}C")
expected_val = 38.00
accuracy_gap = abs(expected_val - prediction_temp)
print("--"* 40)
if accuracy_gap < 0.5:
    print(f"Result: Success! The GRU recognized the +1 trend")
else:
    print("Result: The model is still learning the trend" )
print(f"Difference from Mathematical Truth: {accuracy_gap:.4f}C")
y_temp = np.array([23,28,33])
model_gru = Sequential([
    GRU(32,  input_shape=(3,1)),
    Dense(1)
]) 
model_gru.compile(optimizer='adam', loss='mse')
model_gru.fit(X_temp, y_temp, epochs=200, verbose=0)
test_input =[35, 36, 37]
test_input = np.array([[[35]], [[36]], [[37]]])
prediction_temp = model_gru.predict(test_input)[0][0]

print("\n------GRU Prediction-------")
print(f"Input Sequence (Past 3 Days): {test_input}")
print(f"Model's Predicted Temperature: {prediction_temp:.2f}C")
expected_val = 38.00
accuracy_gap = abs(expected_val - prediction_temp)
print("--"* 40)
if accuracy_gap < 0.5:
    print(f"Result: Success! The GRU recongnized the +1 trend")
else:
    print("Result: The model is still learning the trend" )
print(f"Difference from Mathematical Truth: {accuracy_gap:.4f}C")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 475ms/step

------GRU Prediction-------
Input Sequence (Past 3 Days): [35, 36, 37]
Model's Predicted Temperature: 11.08C
--------------------------------------------------------------------------------
Result: The model is still learning the trend
Difference from Mathematical Truth: 26.9179C
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 405ms/step

------GRU Prediction-------
Input Sequence (Past 3 Days): [[[35]]

 [[36]]

 [[37]]]
Model's Predicted Temperature: 8.87C
--------------------------------------------------------------------------------
Result: The model is still learning the trend
Difference from Mathematical Truth: 29.1342C
